In [ ]:
import numpy as np

In [ ]:
python_list = [1,2,3,4,5]

arr = np.array([1,2,3,4,5])

print(type(python_list))
print(type(arr))
print(arr)


In [ ]:
print(arr * 2)
print(arr + 10)
print(arr.mean())
print(arr.sum())


In [ ]:
matrix = np.array([
    [1,2,3],
    [4,5,6],
    [7,8,9]
])

print(matrix.shape)
print(matrix[0])
print(matrix[0,1])
print(matrix[:,1])



In [ ]:
cat = np.array([0.9,0.1,0.2])
dog = np.array([0.8,0.2,0.1])
car = np.array([0.1,0.9,0.8])


def cosine_similarity (a,b):
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(cosine_similarity(cat,dog))
print(cosine_similarity(cat,car))
print(cosine_similarity(dog,cat))


In [ ]:
# Cell 5 — Pandas basics (handle data like a spreadsheet)

import pandas as pd

# Create a DataFrame — like an Excel table
data = {
    'model': ['gpt-4o', 'claude-3-5-sonnet', 'gemini-1.5-pro', 'llama-3.1-70b'],
    'context_tokens': [128000, 200000, 1000000, 128000],
    'cost_per_1m_input': [5.0, 3.0, 1.25, 0.0],
    'open_source': [False, False, False, True]
}

df = pd.DataFrame(data)
print(df)

In [ ]:
# ---
# Cell 6 — Filtering and querying data

# Filter: only open source models
print(df[df['open_source'] == True])

# Filter: context window > 150k tokens
print(df[df['context_tokens'] > 150000])

# Sort by cost (cheapest first)
print(df.sort_values('cost_per_1m_input'))

# Get one column
print(df['model'])

# Basic stats
print(df['context_tokens'].mean())   # average context window
print(df['context_tokens'].max())    # biggest context window


In [ ]:

# ---
# Cell 7 — Matplotlib: visualize data

import matplotlib.pyplot as plt

models = df['model']
costs = df['cost_per_1m_input']

plt.figure(figsize=(8, 5))
plt.bar(models, costs, color=['#3b82f6', '#8b5cf6', '#10b981', '#f59e0b'])
plt.title('LLM Cost per 1M Input Tokens')
plt.xlabel('Model')
plt.ylabel('Cost ($)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:


# ---
# Cell 8 — Putting it together: simulate RAG similarity search

# Simulate what Qdrant does when you query your RAG chatbot

# 5 "document" embeddings (fake, 4-dimensional for readability)
documents = np.array([
    [0.9, 0.1, 0.2, 0.3],   # doc 0: "Python is great for ML"
    [0.8, 0.2, 0.1, 0.4],   # doc 1: "NumPy makes math fast"
    [0.1, 0.9, 0.8, 0.2],   # doc 2: "Paris is the capital of France"
    [0.2, 0.8, 0.9, 0.1],   # doc 3: "The Eiffel Tower is in Paris"
    [0.85, 0.15, 0.1, 0.35], # doc 4: "Pandas handles tabular data"
])

doc_texts = [
    "Python is great for ML",
    "NumPy makes math fast",
    "Paris is the capital of France",
    "The Eiffel Tower is in Paris",
    "Pandas handles tabular data"
]

# User query embedding (similar to docs 0,1,4 — ML/Python topic)
query = np.array([0.88, 0.12, 0.15, 0.32])

# Calculate similarity to all docs
similarities = []
for i, doc in enumerate(documents):
    sim = cosine_similarity(query, doc)
    similarities.append((sim, doc_texts[i]))

# Sort by similarity (highest first)
similarities.sort(reverse=True)


print(f"similarities -> {similarities}" )

print("Query: 'how do I work with data in Python?'\n")
print("Most relevant documents:")
for score, text in similarities:
    print(f"  {score:.3f} — {text}")



In [ ]:
# Traning first model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

In [ ]:
# Features: [context_tokens, cost_per_1m_input]

X = np.array([
     [128000, 5.0],    # gpt-4o
    [200000, 3.0],    # claude
    [1000000, 1.25],  # gemini
    [128000, 0.0],    # llama (open source)
    [32000, 0.0],     # mistral (open source)
    [70000, 2.0],     # some closed model
])

In [ ]:
# Labels: 0 = closed, 1 = open source

y = np.array([0,0,0,1,1,0])

# Scale features (important - context tokens are huge numbers vs cost)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train model

model = LogisticRegression()
model.fit(X_scaled,y)

# Predict on new model: 64k context $0.50/1M cost

new_model = scaler.transform([[64000,0.5]])
prediction = model.predict(new_model)
probability = model.predict_log_proba(new_model)


print(f"Prediction: {'open source' if prediction[0] == 1 else 'closed'}")
print(f"Confidence: {probability[0].max():.1%}")

In [ ]:
# Cell 10 — Linear Regression: predict cost from context size

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np


# Can we predict cost from context window size?
X = np.array([[128000], [200000], [1000000], [32000], [128000], [70000]])
y = np.array([5.0, 3.0, 1.25, 0.5, 2.0, 1.5])  # cost per 1M tokens


model = LinearRegression()
model.fit(X,y)

# Predict cost for a 500k context model
prediction = model.predict([[500000]])
print(f"Predicted cost for 500k context model: ${prediction[0]:.2f}/1M tokens")
print(f"Model slope: {model.coef_[0]:.8f}")
print(f"Model intercept: {model.intercept_:.2f}")

print(prediction)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

# Bigger dataset — LLM features: [context_tokens, cost, params_billions]
X = np.array([
    [128000, 5.0, 200],   # gpt-4o       — closed
    [200000, 3.0, 100],   # claude        — closed
    [1000000, 1.25, 90],  # gemini        — closed
    [128000, 0.0, 70],    # llama-70b     — open
    [32000, 0.0, 7],      # mistral-7b    — open
    [128000, 0.0, 8],     # llama-3-8b    — open
    [100000, 2.5, 52],    # some closed   — closed
    [64000, 0.0, 13],     # code llama    — open
    [128000, 4.0, 175],   # gpt-3.5       — closed
    [200000, 0.0, 34],    # some open     — open
])
y = np.array([0, 0, 0, 1, 1, 1, 0, 1, 0, 1])

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled,y_train)

predictions = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test,predictions)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Predictions: {predictions}")
print(f"Actual:      {y_test}")
print(f"Accuracy: {accuracy:.1%}")

In [ ]:
print(np.random.seed(42))

In [ ]:
# Cell 12 — Neural Network: build one from scratch to understand what's happening

import numpy as np

# Activation function: sigmoid squashes any number to 0-1
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Same data: predict open source (1) or closed (0)
# Features: [cost_normalized, context_normalized]
X = np.array([
    [1.0, 0.5],   # expensive + medium context = closed
    [0.6, 0.8],   # medium cost + large context = closed
    [0.0, 0.3],   # free + small context = open
    [0.0, 0.5],   # free + medium context = open
])
y = np.array([[0], [0], [1], [1]])  # labels

np.random.seed(42)

# Network: 2 inputs → 3 hidden neurons → 1 output
weights1 = np.random.randn(2, 3)   # input → hidden
weights2 = np.random.randn(3, 1)   # hidden → output

learning_rate = 0.5

# Train for 5000 iterations
for epoch in range(5000):
    # FORWARD PASS — data flows left to right
    hidden = sigmoid(np.dot(X, weights1))
    output = sigmoid(np.dot(hidden, weights2))

    # BACKWARD PASS — error flows right to left, adjusting weights
    output_error = y - output
    output_delta = output_error * sigmoid_derivative(output)

    hidden_error = output_delta.dot(weights2.T)
    hidden_delta = hidden_error * sigmoid_derivative(hidden)

    weights2 += hidden.T.dot(output_delta) * learning_rate
    weights1 += X.T.dot(hidden_delta) * learning_rate

    if epoch % 1000 == 0:
        loss = np.mean(np.abs(output_error))
        print(f"Epoch {epoch:4d} — Loss: {loss:.4f}")

print(f"\nFinal predictions:")
for i, pred in enumerate(output):
    label = "open" if pred[0] > 0.5 else "closed"
    print(f"  Input {X[i]} → {pred[0]:.3f} → {label}")

# What to watch: loss number drops each 1000 epochs. That's learning. The network adjusting weights1 and weights2 is exactly what happens inside every LLM — just at billions-of-parameters scale.

In [ ]:
# Transformers — how attention works (most important concept for LLM depth):

# Cell 13 — Attention mechanism: the core of every LLM

import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

# Sentence: ["The", "cat", "sat"]
# Each word has an embedding (simplified to 4 dimensions)
words = ["The", "cat", "sat"]
embeddings = np.array([
    [0.9, 0.1, 0.2, 0.3],  # "The"
    [0.8, 0.7, 0.1, 0.2],  # "cat"
    [0.2, 0.3, 0.9, 0.8],  # "sat"
])

# Attention asks: for each word, how much should it "attend to" every other word?
# We compute this as dot product similarity between all word pairs

print("=== Attention Scores (before softmax) ===")
scores = embeddings @ embeddings.T  # shape: (3, 3)
print(scores.round(2))

print("\n=== Attention Weights (after softmax — must sum to 1.0) ===")
attention_weights = np.array([softmax(row) for row in scores])
print(attention_weights.round(3))

print("\n=== What each word 'focuses on' ===")
for i, word in enumerate(words):
    focus = {words[j]: f"{attention_weights[i][j]:.2f}" for j in range(len(words))}
    print(f"  '{word}' attends to: {focus}")

print("\n=== New representations (weighted mix of all words) ===")
new_embeddings = attention_weights @ embeddings
for i, word in enumerate(words):
    print(f"  '{word}' new vector: {new_embeddings[i].round(3)}")

# The insight: after attention, each word's vector is now a weighted blend of all other words. "cat" now contains information about "The" and "sat". That's how LLMs understand context — every token sees every other token and decides what's relevant.

# GPT, Claude, Llama — all just this operation, stacked 96+ layers deep.


In [ ]:
import tiktoken
# Cell 14 — Tokenization: LLMs don't read words, they read "tokens"
# A token is roughly 3-4 characters, or a common word fragment

# We'll use tiktoken — OpenAI's tokenizer (same as GPT-4 uses)
# Install first: pip install tiktoken


# Load GPT-4's tokenizer

enc = tiktoken.get_encoding("cl100k_base")

sentences = [
    "Hello world",
    "I am sourov",
    "I love building AI apps!",
    "supercalifragilistic"
]

print("=== How LLMs see your text ===\n")

for sentence in sentences:
        # Convert text → list of token IDs (numbers)
        token_ids = enc.encode(sentence)

        # Convert each token ID back to the text chunk it represents
        token_texts = [enc.decode([t]) for t in token_ids]

        print(f"Text:       '{sentence}'")
        print(f"Token IDs:  {token_ids}")
        print(f"Tokens:     {token_texts}")
        print(f"Count:      {len(token_ids)} tokens")
        print()



# Why this matters: LLMs charge per token, not per word
# "context window" = max tokens, not max words
test = "gpt-4o has a 128,000 token context window"
tokens = enc.encode(test)
print(f"'{test}'")
print(f"= {len(tokens)} tokens (not {len(test.split())} words)")
